# Разработка модели машинного обучения

## Импортирование библиотек

In [1]:
# для работы с датафреймами
import pandas as pd

# для визуализации результатов
import matplotlib.pyplot as plt

# для работы с массивами
import numpy as np

# для преобразования текста
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize

# для работы со строками
import string

# вспомогательные функции
from function import *

# для работы с датасетами
from datasets import Dataset, DatasetDict

# для обработки текста
from pymystem3 import Mystem

# токенизатор и модель
from transformers import T5Tokenizer, T5ForConditionalGeneration
# аргументы для обучения, и трейнер
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
# коллатор
from transformers import DataCollatorForSeq2Seq

# основной модуль для нейронных сетей
import torch

# модуль с метрикой оценивания
import evaluate


c:\Users\user1\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Определяю устройство, на котором будут производиться вычисления:

In [2]:
# получаю девайс
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# вывожу девайс
print(device)

cuda


## Загрузка данных

In [3]:
# загружаю тренировочную выборку
train_dataset = Dataset.from_parquet("Dataset/train.parquet")
# валидационную выборку
validation_dataset = Dataset.from_parquet("Dataset/validation.parquet")
# тестовую выборку
test_dataset = Dataset.from_parquet("Dataset/test.parquet")

# объединяю все выборки в объект Dataset
dataset = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset
})
# вывожу структуру получившегося набора данных
dataset

DatasetDict({
    train: Dataset({
        features: ['text_path', 'annotation_path', 'tags_path', 'text', 'summary', 'tag', 'text_all_symb', 'summary_all_symb', 'tag_all_symb', 'text_clean', 'summary_clean', 'tag_clean', 'text_words', 'summary_words', 'tag_words', 'id', 'processed_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 329
    })
    validation: Dataset({
        features: ['text_path', 'annotation_path', 'tags_path', 'text', 'summary', 'tag', 'text_all_symb', 'summary_all_symb', 'tag_all_symb', 'text_clean', 'summary_clean', 'tag_clean', 'text_words', 'summary_words', 'tag_words', 'id', 'processed_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 41
    })
    test: Dataset({
        features: ['text_path', 'annotation_path', 'tags_path', 'text', 'summary', 'tag', 'text_all_symb', 'summary_all_symb', 'tag_all_symb', 'text_clean', 'summary_clean', 'tag_clean', 'text_words', 'summary_words', 'tag_words', 'id', 'processed_text', 'input_ids', 

## Подбор алгоритма обучения

Блаблабла чоооо швепс эщкере

### Алгоритм обучения

Передо мной стоит задача `суммаризации` текста. Это значит, что надо реализовать нейронную сеть, которая будет находить и выписывать краткое содержание текста. Для этого я буду использовать предобученную модель `"sarahai/ruT5-base-summarizer"`, которую настрою работать на своих данных

### Метрики для суммаризации текста

Для суммаризации одной из наиболее часто используемых метрик является оценка `ROUGE` (сокращение от `Recall-Oriented Understudy for Gisting Evaluation`). Основная идея этой метрики заключается в сравнении сгенерированного текста с набором эталонных текстов, которые обычно создаются людьми. Чтобы сделать ее более точной, предположим, что мы хотим сравнить следующие две строчки:

In [4]:
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"

Одним из способов их сравнения может быть подсчет `количества перекрывающихся слов`, которых в данном случае будет 6. Однако это несколько грубовато, поэтому вместо этого `ROUGE` основывается на вычислении оценок `precision` и `recall` для перекрытия

Для `ROUGE` `recall` измеряет, насколько эталонное резюме соответствует `сгенерированному`. Если мы просто сравниваем слова, `recall` можно рассчитать по следующей формуле:

$$\text{Recall} = \frac{\text{Number of overlapping words}}{\text{Total number of words in reference summary}}$$

Для нашего простого примера выше эта формула дает идеальный `recall` 6/6 = `1`; то есть все слова в эталонном тексте были получены моделью. Это может показаться замечательным, но представьте, если бы сгенерированный нами текст был “I really really loved reading the Hunger Games all night”. Это тоже дало бы идеальный `recall`, но, возможно, было бы хуже, поскольку было бы многословным. Чтобы справиться с этими сценариями, мы также вычисляем `precision`, которая в контексте `ROUGE` измеряет, насколько `сгенерированное` резюме было `релевантным`:

$$\text{Precision} = \frac{\text{Number of overlapping words}}{\text{Total number of words in generated summary}}$$

Если применить это к нашему подробному тексту, то `precision` составит 6/10 = 0,6, что значительно хуже, чем `precision` 6/7 = 0,86, полученная при использовании более короткого текста. На практике обычно вычисляют и `precision`, и `recall`, а затем `F1-score` (среднее гармоническое из `precision` и `recall`)

Загружаю метрику `ROUGE`:

In [5]:
# загружаю используемую метрику
rouge_score = evaluate.load('rouge')

Затем мы можем использовать функцию `rouge_score.compute()`, чтобы рассчитать все метрики сразу:

In [6]:
# считаем метрики примеров
scores = rouge_score.compute(
    predictions=[generated_summary],
    references=[reference_summary]
)
# вывожу получившиеся метрики
scores

{'rouge1': np.float64(0.923076923076923),
 'rouge2': np.float64(0.7272727272727272),
 'rougeL': np.float64(0.923076923076923),
 'rougeLsum': np.float64(0.923076923076923)}

## Работа с нейронной сетью

Для начала, надо инициализировать саму `модель` и ее `токенизатор`:

In [7]:
# имя модели
model_name = "sarahai/ruT5-base-summarizer"
# инициализируем модель
model = T5ForConditionalGeneration.from_pretrained(model_name)
# инициализирую токенизатор
tokenizer = T5Tokenizer.from_pretrained(model_name)

Теперь, надо назначить вычислительное устройство для модели (его я определил выше):

In [8]:
# назначаю устройство и вывожу архитектуру модели
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

Дообучение `ruT5` с `API` `Trainer`

In [9]:
# назначаю кол-во батчей
batch_size = 8
# кол-во эпох
epochs = 20
# функция потерь
logging_steps = len(dataset['train']) // batch_size
name = model_name.split('/')[-1]
print(name)

ruT5-base-summarizer


Теперь, нужно создать объект `Seq2SeqTrainingArguments`, чтобы заполнить гиперпараметры модели:

In [10]:
# объект с гиперпараметрами
args = Seq2SeqTrainingArguments(
    output_dir='model-4-summary',
    overwrite_output_dir=True,
    eval_strategy='epoch',                     # оценка после каждой эпохи
    per_device_train_batch_size=batch_size,     # кол-во тренировочных батчей
    per_device_eval_batch_size=batch_size,      # кол-во батчей для оценки
    gradient_accumulation_steps=2,              # кол-во шагов накопления градиента до обновления
    torch_empty_cache_steps=4,                  # oчистка кэша GPU через каждые 4 шага
    learning_rate=1e-4,                         # скорость обучения
    num_train_epochs=epochs,                    # кол-во эпох
    logging_steps=logging_steps,                # частота логов
    seed=42,                                    # сид для воспроизводимости результатов
    fp16=True,                                  # bbспользование mixed precision для ускорения обучения
    weight_decay=0.01,                          # отложенные весаL2-регуляризация для предотвращения переобучения
    optim='adamw_torch',                        # оптимизатор
    report_to="tensorboard",

)

Следующее, что нужно сделать, это предоставить тренеру функцию `compute_metrics()`, чтобы оценить нашу модель во время обучения. Для суммаризации это немного сложнее, чем просто вызвать `rouge_score.compute()` для прогнозов модели, поскольку нужно декодировать выводы и метки в текст, прежде чем вычислить оценку `ROUGE`. Следующая функция делает именно это, а также использует функцию `sent_tokenize()` из `nltk` для разделения предложений резюме символом новой строки

In [11]:
# функция для вычисления метрик
def compute_metrics(eval_pred):
    # получаем предсказания и их метки
    predictions, labels = eval_pred
    # print(f"Predictions {predictions}, shape: {predictions[0].shape}, type: {type(predictions)}")

    # print(f"Labels {labels}, shape: {labels.shape}, type: {type(labels)}")
    # Если predictions — это кортеж, берем первый элемент (логи)
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Преобразуем логи в индексы токенов с помощью argmax
    predicted_token_ids = np.argmax(predictions, axis=-1)
    # декодируем сгенерированные суммаризации в текст
    decoded_preds = tokenizer.batch_decode(predicted_token_ids, skip_special_tokens=True)
    # заменяем -100 в метках, тк декодировать их нельзя
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # декодируем эталонные изложения 
    decoded_summary = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # ROUGE ожидает символ новой строки после каждого предложения
    decoded_preds = ['\n'.join(sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_summary = ['\n'.join(sent_tokenize(label.strip())) for label in decoded_summary]
    
    # вычисляем метрики ROUGE
    result = rouge_score.compute(
        predictions=decoded_preds,
        references=decoded_summary,
        use_stemmer=True            # проверить и с ним, и без него
    )

    # получаем оценки
    result = {k: v*100 for k, v in result.items()}
    return {k: round(v,4) for k,v in result.items()}

Также, для обучения модели необходим `коллатор`, который будет сдвигать метки на 1 каждый шаг

In [12]:
# инициализируем коллатор
collator = DataCollatorForSeq2Seq(tokenizer, model)

Получаем датасет для обучения модели

In [13]:
# убираем лишние колонки
train_data = dataset.select_columns(['input_ids', 'attention_mask', 'labels'])
# выводим датасет
train_data

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 329
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 41
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 42
    })
})

In [14]:
type(train_data['train']['input_ids'][0])

list

Теперь, надо создать объект `Trainer` для запуска обучения модели

In [15]:
# объект trainer 
trainer = Seq2SeqTrainer(
    model,                                  # модель
    args,                                   # тренировочные аргументы
    train_dataset=train_data['train'],      # датасет для обучения модели
    eval_dataset=train_data['validation'],  # датасет для оценки модели
    data_collator=collator,                 # коллатор
    processing_class=tokenizer,             # токенизатор
    compute_metrics=compute_metrics         # функция для вычисления метрик
)

После этого, можно начать обучение модели

In [16]:
torch.cuda.empty_cache()

In [17]:
# запуск обучения модели
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,1.674144,11.673700,2.613200,10.984500,11.206400
2,1.940100,1.652601,11.947700,2.604400,11.356800,11.510600
3,1.940100,1.640401,10.087300,2.599000,9.556400,9.556400
4,1.743000,1.632263,9.560500,2.578400,9.227900,9.231400
5,1.743000,1.629171,9.479200,2.578400,9.083700,9.083700
6,1.674700,1.629797,9.741800,2.578400,9.417800,9.388000


TrainOutput(global_step=126, training_loss=1.7843189542255704, metrics={'train_runtime': 1549.1312, 'train_samples_per_second': 1.274, 'train_steps_per_second': 0.081, 'total_flos': 1202082875965440.0, 'train_loss': 1.7843189542255704, 'epoch': 6.0})

Сохраняю полученную модель

In [18]:
# Сохранение модели
model.save_pretrained("./saved_model")

# Сохранение токенизатора
tokenizer.save_pretrained("./saved_model")

('./saved_model\\tokenizer_config.json',
 './saved_model\\special_tokens_map.json',
 './saved_model\\spiece.model',
 './saved_model\\added_tokens.json')

In [20]:
# Загрузка модели
model = T5ForConditionalGeneration.from_pretrained("./saved_model")

# Загрузка токенизатора
tokenizer = T5Tokenizer.from_pretrained("./saved_model")

In [21]:
device = torch.device('cpu')
print(device)
model.to(device)

cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [22]:
rouge_score = evaluate.load('rouge')

In [28]:
index = 4
orig_text = dataset['validation']['text'][index]
summary = dataset['validation']['summary'][index]

In [29]:
input_ids = tokenizer(orig_text, return_tensors='pt').input_ids.to(device)
outputs = model.generate(input_ids, max_length=100, num_beams=4)
gen_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

In [30]:
scores = rouge_score.compute(
    predictions=[gen_summary], references=[summary]
)
pd.DataFrame.from_dict([scores])

,rouge1,rouge2,rougeL,rougeLsum
0,0.0,0.0,0.0,0.0


In [37]:
def get_summary(text: str):
    input_ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
    outputs = model.generate(input_ids, max_length=80, num_beams=4)
    gen_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f'>> Original Text: {text}')

    print(f'\n\n\n>> Generated summary: {gen_summary}')

In [38]:
text = '''«Пила» (Saw) — культовая хоррор-кинофраншиза в жанре триллера и слэшера, начавшаяся в 2004 году. Серия фильмов рассказывает о гениальном, но жестоком маньяке Джоне Крамере, известном как Конструктор (Jigsaw). Он создаёт изощрённые ловушки, чтобы заставить своих жертв «ценить жизнь» через мучительные испытания.

Франшиза знаменита своей запутанной хронологией, неожиданными сюжетными поворотами и кровавыми сценами. Всего вышло 10 фильмов, последний из которых — «Пила 10» (2023). Несмотря на смерть главного антагониста в ранних частях, его «наследие» продолжали последователи.

«Пила» стала одной из самых прибыльных хоррор-серий, породив множество теорий, мемов и культурных отсылок.'''

In [39]:
get_summary(text)

>> Original Text: «Пила» (Saw) — культовая хоррор-кинофраншиза в жанре триллера и слэшера, начавшаяся в 2004 году. Серия фильмов рассказывает о гениальном, но жестоком маньяке Джоне Крамере, известном как Конструктор (Jigsaw). Он создаёт изощрённые ловушки, чтобы заставить своих жертв «ценить жизнь» через мучительные испытания.

Франшиза знаменита своей запутанной хронологией, неожиданными сюжетными поворотами и кровавыми сценами. Всего вышло 10 фильмов, последний из которых — «Пила 10» (2023). Несмотря на смерть главного антагониста в ранних частях, его «наследие» продолжали последователи.

«Пила» стала одной из самых прибыльных хоррор-серий, породив множество теорий, мемов и культурных отсылок.



>> Generated summary: Хоррор-кинофраншиза «Пила» (Saw) рассказывает о жестоком маньяке Джоне Крамере, известном как Конструктор (Jigsaw), который создаёт изощрённые ловушки, чтобы заставить своих жертв «ценить жизнь».


In [31]:
print(f'>> Original Text: {orig_text}')
print(f'\n\n\n>> Summary: {summary}')
print(f'\n\n\n>> Generated summary: {gen_summary}')

>> Original Text: Человечество в очередной раз входит в
эпоху глобальных перемен, на этот раз в эпоху
цифровизации. Хотя каждое поколение, конечно же, может заявить о своей эпохе Великих
перемен и/или потрясений, да и не об одной.
В современном мире, где огромную роль
играет формирующаяся цифровая культура, все
сферы социального бытия и социальной практики, многие области знания трансформируются под воздействием инфокоммуникативных
технологий.
Простое использование цифровых технологий в гуманитарных науках, в первую очередь в философии, вызывает сложности не
только методологического и методического характера, но и неясности в самом определении
области применения.
Исследователь Е. Елькина (2020) подчеркивает, что «Технологическая гонка, знаменующая переход ведущих экономических держав
в шестой технологический уклад, породила
большой поток терминов, в которых «дигитальность» рассматривается как маркер изменений
предметных областей, где эти технологии используются («цифровая экономика», «

## Анализ эффективности модели

## Оптимизация

## Разработка программного продукта